# Using the KPoEM Dataset and Model: Poetry Generation

- **KPpEM Emoiton Classification Medel** : AKS-DHLAB. (2025). KPoEM  [Computer software]. Hugging Face. https://doi.org/10.57967/hf/6301 
- This code is uploaded in AKS-DHLAB. (2025). KPoEM [Computer software]. GitHub. https://github.com/AKS-DHLAB/KPoEM  

## 1. Basic setup and library imports
- Import Libraries and Set Configuration

In [1]:
!pip install -q huggingface_hub langchain langchain-core langchain-community langchain-huggingface 
!pip install faiss-cpu

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/bitsandbytes-0.45.4.dev0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.12.0.dev0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.23a0+6627725-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1

In [2]:
# 라이브러리 임포트
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, ElectraModel, AutoModelForCausalLM, pipeline
from huggingface_hub import hf_hub_download
import os
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# from langchain.chains import LLMChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.embeddings.base import Embeddings
from langchain_community.vectorstores import FAISS
import langchain
import langchain_community
import langchain_core
import langchain_huggingface

## 2. Loading the KPoEM Model

In [3]:
# 기초 세팅
REPO_ID = "AKS-DHLAB/KPoEM" # 허깅페이스에 업로드된 감정분류모델 id
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #GPU 사용
THRESH_HOLD = 0.3

In [4]:
# KPoEM_Classifier 클래스
class KPoEM_Classifier(nn.Module):
    def __init__(self, repo_id, device):
        self.labels = [
            '불평/불만', '환영/호의', '감동/감탄', '지긋지긋', '고마움', '슬픔', '화남/분노', '존경',
            '기대감', '우쭐댐/무시함', '안타까움/실망', '비장함', '의심/불신', '뿌듯함', '편안/쾌적',
            '신기함/관심', '아껴주는', '부끄러움', '공포/무서움', '절망', '한심함', '역겨움/징그러움',
            '짜증', '어이없음', '없음', '패배/자기혐오', '귀찮음', '힘듦/지침', '즐거움/신남', '깨달음',
            '죄책감', '증오/혐오', '흐뭇함(귀여움/예쁨)', '당황/난처', '경악', '부담/안_내킴', '서러움',
            '재미없음', '불쌍함/연민', '놀람', '행복', '불안/걱정', '기쁨', '안심/신뢰'
        ]
        num_labels = len(self.labels)
        #모델 & 토크나이저 로드
        super().__init__()
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(repo_id) 
        self.electra = AutoModel.from_pretrained(repo_id)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.1),
            nn.Linear(self.electra.config.hidden_size, num_labels)
        )

        weights_path = hf_hub_download(repo_id=repo_id, filename="classifier_state.bin") #가중치 불러오기
        self.classifier.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.to(self.device)
        self.eval()

    # 텍스트를 입력받아 최종 logits 반환
    def forward(self, text: str):
        encoding = self.tokenizer(
          text,
          add_special_tokens=True,
          max_length=512,
          padding="max_length",
          truncation=True,
          return_tensors='pt',
        ).to(self.device)

        with torch.no_grad():
            outputs = self.electra(
                input_ids=encoding["input_ids"],
                attention_mask=encoding["attention_mask"],
                token_type_ids=encoding["token_type_ids"]
            )

        pooled_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled_output)
        return logits

    def analyze(self, text: str, threshold=0):
        logits = self.forward(text)
        probabilities = torch.sigmoid(logits.squeeze()) #확률로 변환 → threshold 이상이면 선택
        predictions = (probabilities > threshold).int()

        result_dict = {
            self.labels[i]: float(round(probabilities[i].item(), 3))
            for i, label_id in enumerate(predictions)
            if label_id == 1
        }

        # 확률값 기준 내림차순 정렬된 dict로 반환
        result_dict = dict(sorted(result_dict.items(), key=lambda x: x[1], reverse=True))
        return result_dict


In [5]:
# KPoEM 모델 로드
print(f"... '{DEVICE}' 환경에서 '{REPO_ID}' 모델을 로드하고 있습니다 ...")
kpoem_model = KPoEM_Classifier(repo_id=REPO_ID, device=DEVICE)
print("KPoEM 모델을 성공적으로 로드하였습니다.")

... 'cuda' 환경에서 'AKS-DHLAB/KPoEM' 모델을 로드하고 있습니다 ...
KPoEM 모델을 성공적으로 로드하였습니다.


In [6]:
# 모델 사용 테스트
test = """흙에서 자란 내 마음
파아란 하늘 빛이 그립어
함부로 쏜 화살을 찾으려
풀섶 이슬에 함추름 휘적시든 곳,

― 그 곳이 참하 꿈엔들 잊힐 리야.

전설바다에 춤추는 밤불결 같은
검은 귀밑머리 날리는 어린 누이와
아무렇지도 않고 여쁠 것도 없는
사철 발벗은 안해가
따가운 해ㅅ살을 등에 지고 이삭 줏던 곳,

― 그 곳이 참하 꿈엔들 잊힐 리야."""
result = kpoem_model.analyze(test, threshold=THRESH_HOLD)
result

{'슬픔': 0.969,
 '서러움': 0.94,
 '안타까움/실망': 0.902,
 '불쌍함/연민': 0.749,
 '불안/걱정': 0.679,
 '아껴주는': 0.639,
 '힘듦/지침': 0.626,
 '기대감': 0.424,
 '절망': 0.336,
 '깨달음': 0.324,
 '부담/안_내킴': 0.314}

## 3. Downloading and Loading the LLM for Poetry Generation

In [7]:
MODEL_ID = "K-intelligence/Midm-2.0-Base-Instruct"
CACHE_DIR = "/home/work/KPoEM/code/AICreation/model"

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

#모델 로드 - device_map="auto" 유지 (자동 분산 로드)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",           # GPU + CPU 메모리 자동 분산
    low_cpu_mem_usage=True,      # 로딩 시 CPU 메모리 절약
    trust_remote_code=True,
    cache_dir=CACHE_DIR
)

print("모델 로드 완료. GPU/CPU 자동 오프로딩 활성화")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

모델 로드 완료. GPU/CPU 자동 오프로딩 활성화


In [8]:
# HuggingFace pipeline 생성 (device 설정하지 않아도 자동)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # do_sample=False,
    temperature=0.7,
    top_p=0.9,
    max_new_tokens=128,
    repetition_penalty=1.2
)

# LangChain LLM 래퍼로 감싸기
llm = HuggingFacePipeline(pipeline=pipe)


Device set to use cuda:0


In [9]:
# 간단한 프롬프트 템플릿
prompt = PromptTemplate(
    input_variables=["topic"],
    template="다음 주제로 감성적인 시를 써주세요:\n주제: {topic}\n\n### 시:\n"
)
parser = StrOutputParser()
# LLMChain 생성
chain = prompt | llm | parser

result = chain.invoke({"topic": "겨울바람"})
print(result)

다음 주제로 감성적인 시를 써주세요:
주제: 겨울바람

### 시:
겨울 바람이 스치면
얼어붙은 마음에 온기가 깃든다.
눈 덮인 길 위, 쓸쓸한 발걸음마다
희망의 씨앗을 심어준다.

차가운 숨결 속에서도,
사랑과 꿈의 노래는 계속된다.
하얀 세상 아래 숨겨진 이야기들이
따뜻한 가슴으로 스며들어간다.


## 4. Prompt-Based Poetry Generation 1
Using Input Text and Emotion Classification Results (Without a Vector Database)

### 감정 분류 모델만 적용하여 LLM으로 시 생성(Vector DB 미적용)

In [8]:
def extract_poetry_section(template):
    # Split the template by "### 시:" and extract the part after it
    if "### 시:" in template:
        poetry_section = template.split("### 시:")[1].strip()
        # Split by lines and return as a list
        poetry_lines = poetry_section.splitlines()
        return poetry_lines
    else:
        return None

In [9]:
# 5️⃣ 전체 흐름 함수
def emotion_to_poetry(sample_text): #감정 분류에 사용할 사용자 텍스트 인풋.
    emotion_scores = kpoem_model.analyze(sample_text, threshold=0.3)
    
    template = """
    ### 시스템:
    당신은 창의적이고 감성적인 근현대 시인입니다.
    아래에 제시된 원문 텍스트와 감정 목록을 참고하여 시를 지으세요.
    원문 텍스트는 시의 소재나 분위기를 떠올리는 데 활용하고,
    감정 목록에 언급된 감정들을 시의 핵심 정서로 반영하세요.
    
    영어나 다른 언어는 사용하지 말고, 한국어로만 작성하세요.
    이모지나 그림은 사용하지 마세요.
    한국 고유의 표현을 사용하고,
    은유와 상징을 통해 창의적으로 감정을 표현하세요.
    시 해설은 필요 없습니다. 시만 작성하세요.
    
    ### 원문 텍스트:
    {sample_text}
    
    ### 감정 목록:
    {emotion}
    
    ### 시:
    """

    # """ ### 시스템: 당신은 창의적이고 감성적인 근현대 시인입니다. 다음 감정목록에 언급된 감정들을 주된 시의 정서로 활용하세요. 영어나 다른 언어는 사용하지 말고, 한국어로만 작성하세요. 이모지나 그림을 사용하지 마세요. {context_snippets} 위 문장들에서 옛스러운 한국 고유의 표현을 사용하여 시를 하나 지어주세요. 은유와 상징을 사용하여 창의적으로 감정을 표현하세요. 시 해설은 필요없습니다. 시만 써주세요. 감정 목록: {top_emotion} ### 시: """

    prompt = PromptTemplate(
        input_variables=["emotion"],
        template=template.strip()
    )
    # print(emotion_scores)
    chain = LLMChain(llm=llm, prompt=prompt)
    result = chain.run(sample_text=sample_text, emotion=emotion_scores)
    
    return result

In [10]:
sample_text = """미풍에 웃는 아침을 기원하련다"""

In [13]:
# # 6️⃣ 테스트
generated_poem = emotion_to_poetry(sample_text)

In [14]:
print("생성된 시:\n")
extract_poetry_section(generated_poem)

생성된 시:



['아침 창가에 앉은 그대',
 '미풍이 살며시 불어오는 소리 들으며',
 '웃고 있는 내 마음 속 작은 꽃 하나',
 '',
 '어제보다 더 밝은 오늘이여',
 '그대가 내게 준 기쁨처럼',
 '햇살도 덩달아 춤추네',
 '',
 '꽃잎 사이 스며드는 바람결마다',
 '설레임 가득한 기다림 있어라',
 '희망이라는 이름의 푸른 날개짓',
 '',
 '내일이면 또 만날 수 있겠지',
 '그리운 눈빛 마주하며 웃을 그날까지',
 '가슴속 깊이 새긴 약속',
 '',
 '그대 향한 이 마음 끝닿도록',
 '영원히 이어질 우리 이야기여',
 '바람 따라 흐르는 행복 노래하리라']

## 5. Prompt-Based Poetry Generation 2
Using Input Text and Emotion Classification Results (With a Vector Database)

#### - Loading the Vector Store Built with the KcELECTRA Backbone Model

In [10]:
# KcELECTRA 기반 커스텀 임베딩 클래스 정의
class KcELECTRAEmbeddings(Embeddings):
    def __init__(self, model_name: str = "beomi/KcELECTRA-base", device: str = "cpu"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.device = device

    def _embed(self, text: str):
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :]
        return cls_embedding.squeeze().cpu().numpy()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(text).tolist() for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text).tolist()

In [12]:
# 3️⃣ 벡터 임베딩 모델 로딩 (한국어 지원하는 모델 권장) KcElectra -> backbone 모델로 사용
embedding_model = KcELECTRAEmbeddings()

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/514 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [13]:
# 로컬에서 로드 (신뢰할 수 있는 파일일 경우)
vectorstore = FAISS.load_local(
    "./data/poetry_vectorstore", #깃허브 파일은 링크 수정 "./vectorstore" 참고 - https://github.com/AKS-DHLAB/KPoEM/blob/main/KPoEM_vectorstore.ipynb
    embedding_model, 
    allow_dangerous_deserialization=True
)

#### - After Computing Vector Similarity, 
Use the Emotion Information Stored in the Metadata of the Retrieved 100 Entries to Construct Context for Poetry Generation

In [14]:
class PoetryGenerator:
    """감정 분석 + 문맥 검색 + 시 생성 전체 파이프라인"""

    def __init__(self, kpoem_model, vectorstore, llm):
        """
        Args:
            kpoem_model: KPoEM 감정 분류 모델
            vectorstore: FAISS 또는 Chroma 벡터스토어
            llm: LangChain LLM 객체 (HuggingFacePipeline)
        """
        self.kpoem_model = kpoem_model
        self.vectorstore = vectorstore
        self.llm = llm

    # ---------------------------------------------------------------------
    def get_top_emotions(self, scores, top_n=10):
        """점수 dict에서 상위 N개 감정 반환"""
        top = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n])
        return top, list(top.keys())

    # ---------------------------------------------------------------------
    def filter_context_by_emotion(self, docs, top_keys, top_k=10):
        """문맥 리스트(context) 중 감정 교집합이 많은 순으로 상위 top_k 반환"""
        results = []

        for doc in docs:
            doc_emotions = set(doc.metadata.get("emotion", {}).keys())
            overlap = doc_emotions.intersection(top_keys)

            if overlap:
                results.append((doc, len(overlap)))

        # 교집합 개수 많은 순으로 정렬
        results.sort(key=lambda x: x[1], reverse=True)

        return [doc for doc, _ in results[:top_k]]

    # ---------------------------------------------------------------------
    def build_prompt_template(self):
        """LangChain 최신 버전용 PromptTemplate"""
        return PromptTemplate(
            input_variables=["sample_text", "context_snippets", "top_emotion"],
            template="""
        ### 시스템:
        당신은 창의적이고 감성적인 근현대 시인입니다.  
        아래 조건을 모두 만족하는 시를 작성하세요.

        - 한국어로만 작성할 것  
        - 이모지나 그림 없이 순수 시어만 사용할 것  
        - 원문 텍스트는 시의 소재·상징·분위기 아이디어로 활용  
        - 참고 문장들에서는 표현 방식과 어휘의 결을 배우되 그대로 베끼지 말 것  
        - 감정 목록은 시 전체의 정서적 축으로 삼을 것  
        - 제목을 지을 것
        - 해설 금지, 시만 작성  

        ---

        **원문 텍스트:**  
        {sample_text}

        ---

        **참고 문장들:**  
        {context_snippets}

        ---

        **핵심 감정 목록:**  
        {top_emotion}

        ---

        ### 시:
        """.strip()
        )

    # ---------------------------------------------------------------------
    def generate_poetry(self, user_input, top_n=10, context_k=100, filtered_k=10):
        """감정 분석 → 문맥 검색 → 시 생성까지 전체 수행"""

        # 1. 감정 분석
        scores = self.kpoem_model.analyze(user_input, threshold=0.3)
        top_emotions, top_keys = self.get_top_emotions(scores, top_n=top_n)

        # 2. RAG 유사 문맥 검색
        context_candidates = self.vectorstore.similarity_search(user_input, k=context_k)
        filtered = self.filter_context_by_emotion(context_candidates, top_keys, filtered_k)

        # 3. 문맥 텍스트 구성
        context_text = "\n".join(doc.page_content for doc in filtered)

        # 4. 감정 문자열 변환 (LLM에 dict 넣으면 깨짐)
        emotion_text = "\n".join([f"- {e}" for e in top_keys])
        print("문맥, 감정 비슷한 벡터 :", filtered)
        print("텍스트 감정:", top_emotions)
        print("최종문맥정보:", context_text)
        # 5. Prompt + LLM + Parser 연결
        parser = StrOutputParser()
        prompt = self.build_prompt_template()
        chain = prompt | self.llm | parser

        # 6. 실행
        result = chain.invoke({
            "sample_text": user_input,
            "context_snippets": context_text,
            "top_emotion": emotion_text
        })

        return result


In [15]:
# 초기화
generator = PoetryGenerator(kpoem_model=kpoem_model, vectorstore=vectorstore, llm=llm)

In [20]:
# 테스트 실행
user_text = """내가 그의 이름을 불러주기 전에는
그는 다만
하나의 몸짓에 지나지 않았다.

내가 그의 이름을 불러 주었을 때
그는 나에게로 와서
꽃이 되었다."""

In [21]:
# 시 생성
poem = generator.generate_poetry(user_text, top_n=10, context_k=100, filtered_k=5)
print(poem)

문맥, 감정 비슷한 벡터 : [Document(id='d50f6b30-d9eb-4c57-b85b-24a41a5bf144', metadata={'emotion': {'슬픔': 0.6, '비장함': 0.6, '서러움': 0.6, '감동/감탄': 0.4, '고마움': 0.4, '아껴주는': 0.4, '깨달음': 0.4, '불쌍함/연민': 0.4, '환영/호의': 0.2, '화남/분노': 0.2, '존경': 0.2, '기대감': 0.2, '뿌듯함': 0.2, '기쁨': 0.2}, 'poet': '한용운'}, page_content='저리고 쓰린 슬픔은 힘이 되고 열이 되어서 어린 양(羊)과 같은 작은 목숨을 살아 움직이게 합니다'), Document(id='53958e77-8068-499f-9bcd-034bff9add22', metadata={'emotion': {'패배/자기혐오': 0.8, '슬픔': 0.4, '기대감': 0.4, '불쌍함/연민': 0.4, '감동/감탄': 0.2, '고마움': 0.2, '비장함': 0.2, '아껴주는': 0.2, '한심함': 0.2, '힘듦/지침': 0.2, '깨달음': 0.2, '죄책감': 0.2, '서러움': 0.2, '기쁨': 0.2}, 'poet': '이상'}, page_content='통화구를 손바닥으로 꼭 막으면서 내가 죽으면 앉았다 일어서듯이 나비도 날아가리라.'), Document(id='8209bb27-a51e-4bba-b062-5427d03b6649', metadata={'emotion': {'감동/감탄': 0.8, '흐뭇함(귀여움/예쁨)': 0.8, '기쁨': 0.8, '신기함/관심': 0.6, '존경': 0.4, '행복': 0.4, '환영/호의': 0.2, '고마움': 0.2, '기대감': 0.2, '뿌듯함': 0.2, '아껴주는': 0.2, '즐거움/신남': 0.2, '깨달음': 0.2, '놀람': 0.2}, 'poet': '한용운'}, page_content='혀끝에서 물결이 솟고 붓 아래에 꽃이 피어요.'

In [22]:
def extract_poem(text: str) -> str: 
    """ text에서 "### 시:" 이후 나오는 부분만 추출하고, 그 다음 섹션 마커(예: ### 전체:)가 나오면 그 앞까지만 반환. """ 
    start_marker = "### 시:" 
    end_markers = ["### 전체:", "### 해설:", "### 요약:", "#### 시"] # 필요한 경우 확장 가능 
    if start_marker not in text: return "[시를 찾을 수 없습니다]" # 1) 시 시작 부분만 추출 
    poem_section = text.split(start_marker, 1)[1].strip() # 2) 끝 마커가 있으면 그 앞까지만 자르기 
    for end_marker in end_markers: 
        if end_marker in poem_section: 
            poem_section = poem_section.split(end_marker, 1)[0].strip() 
            break # 가장 먼저 등장한 마커까지만 사용 
        return poem_section

In [23]:
extracted_poem = extract_poem(poem)
print(extracted_poem)

<이름 부르는 노래>

어둠 속 한 점이던 존재여,
너의 형체 없던 모습이여,
나는 그대의 이름조차 몰랐노라.

무심히 지나치던 시간 속에 
그대와 나는 서로 알지 못한 채
각자의 길 위에서 홀로 서 있었네.

하지만 어느 순간 문득 찾아온 마음의 떨림,
깊은 밤 고요히 들려오는 속삭임처럼
나의 언어로 너를 부르기 시작했구나.

아, 그때부터였을까
말 없는 침묵이었던 네게서
작은 빛줄기 하나 움터 오르는 것이 보이고
나라는 거울 안에서 처음으로 
완전한 형상을 갖추는 모습이었으리.

이름이란 건
